# 10 · OKF as an MCP server

Stage 2 of the backend. The `okf` package from `09` becomes an **MCP server**: a separate program that offers your pipeline as **tools** any MCP client can call — the LangGraph agents (stage 3), the MCP Inspector, Claude Desktop.

```
 MCP client (this notebook, later the agents)            MCP server (okf_mcp/server.py)
 ───────────────────────────────────────────            ───────────────────────────────
  initialize            ──── "hello, I speak MCP" ────►   "hello, I'm okf, here's what I support"
  tools/list            ──── "what can you do?" ──────►   search_okf, ask_okf, get_concept (+ schemas)
  tools/call search_okf ──── {"question": "..."} ─────►   okf.search(...)  ──► passages as JSON
```

Those three messages are the whole protocol for tools. Everything is JSON (JSON-RPC) sent over a **transport**:
- **streamable HTTP**: the server runs on its own at `http://127.0.0.1:8000/mcp`; many clients can connect. **We use this**: three agents will share one OKF server.
- **stdio**: the client starts the server as a child process and talks through its stdin/stdout. Claude Desktop uses this (config at the end).

**The server does no thinking.** It runs your code when asked. Deciding *when* to call a tool is the client's job (in stage 3: the agent graph).

## 1. The server, piece by piece

All of it is in `okf_mcp/server.py` (your steps 1–4 plus the rest). The important parts:

**The server object and the pipeline, loaded once**
```python
mcp = FastMCP("okf", instructions="Search and answer questions about five AI research papers ...")

@lru_cache(maxsize=1)
def get_okf() -> OKF:          # OKF.load() takes seconds: do it once per server process, reuse after
    return OKF.load()
```

**A tool = a decorated function.** FastMCP reads everything it needs from the Python you already wrote:
```python
@mcp.tool()
def search_okf(question: Annotated[str, Field(description="The question or keywords to search for")]) -> list[Passage]:
    """Find the passages from the five papers that best match a question (no LLM, about 1-2 s)."""
    return get_okf().search(question)
```
| your Python | what the client sees |
|---|---|
| function name | tool name `search_okf` |
| docstring | tool description: what a **model** reads to decide *when* to use it |
| type hint + `Field(description=...)` | **input schema** |
| return type `list[Passage]` (pydantic) | **output schema** |

**Return only what the caller needs.** `ask_okf` returns a small `OKFAnswer` (answer, refused, sources, model, seconds), not the full `Answer` with its 18,000-character context. A tool's result lands in the calling agent's context window, and qwen has 8K.

**Errors are answers too.** `get_concept` raises `ValueError` for an unknown id; the client receives `isError: True` with the message. Write messages *for a model*: say what to do instead.

**A resource** (`okf://status`) is data an app reads by address, like a read-only file. Tools are actions a model chooses; resources are things an app looks up.

**Startup** loads the pipeline, then sends one tiny prompt to Ollama so qwen is already in VRAM (otherwise the first question waits ~90 s), then starts listening. Logs go to **stderr**, because in stdio mode stdout *is* the message channel.

## 2. Start the server

Normally you'd run `python -m okf_mcp.server` in a terminal and leave it open. Here the notebook starts it as a **background process** for you, waits until it says `ready`, and writes its log to `artifacts/logs/mcp_server.log`.

Needs: `pip install -e ".[mcp]"` once (installs `mcp` and `langchain-mcp-adapters`), Ollama running, `.env` with the Qdrant keys.

In [2]:
import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

PORT = 8000
URL = f"http://127.0.0.1:{PORT}/mcp"
LOG = Path("artifacts/logs/mcp_server.log")
LOG.parent.mkdir(parents=True, exist_ok=True)


def port_in_use(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0


if port_in_use(PORT):
    print(f"something is already listening on {PORT}: probably the server from an earlier run. Using it.")
    server = None
else:
    log_file = LOG.open("w", encoding="utf-8")
    server = subprocess.Popen([sys.executable, "-m", "okf_mcp.server", "--port", str(PORT)],
                              stdout=log_file, stderr=subprocess.STDOUT,
                              env={**os.environ, "NO_COLOR": "1", "TERM": "dumb", "PYTHONUNBUFFERED": "1"})
    start = time.perf_counter()
    while "ready" not in LOG.read_text(encoding="utf-8", errors="ignore"):
        if server.poll() is not None:
            raise RuntimeError("the server stopped while starting:\n" + LOG.read_text(encoding="utf-8", errors="ignore")[-2000:])
        if time.perf_counter() - start > 300:
            raise TimeoutError("server not ready after 5 minutes; see " + str(LOG))
        time.sleep(1)
    while not port_in_use(PORT):          # 'ready' is printed just before it starts listening
        time.sleep(0.5)
    print(f"server up in {time.perf_counter() - start:.0f} s (pid {server.pid})\n")
if LOG.exists() and server is not None:
    print(LOG.read_text(encoding="utf-8", errors="ignore")[-1500:])
else:
    print("server is running in your terminal: its log is in that window")

something is already listening on 8000: probably the server from an earlier run. Using it.
server is running in your terminal: its log is in that window


## 3. Connect as a client: handshake + "what can you do?"

`mcp_session()` opens a connection and does the handshake (`initialize`). Jupyter lets you use `await` directly in a cell, so the async MCP client works here as-is.

The timeouts are generous on purpose: an answer can take ~10 s, longer if Ollama has to reload the model.

In [3]:
from contextlib import asynccontextmanager

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


@asynccontextmanager
async def mcp_session(url=URL):
    async with streamablehttp_client(url, timeout=60, sse_read_timeout=900) as (read, write, _):
        async with ClientSession(read, write) as session:
            info = await session.initialize()                        # message 1: the handshake
            yield session, info


async with mcp_session() as (session, info):
    print("connected to:", info.serverInfo.name, "| protocol", info.protocolVersion)
    print("instructions:", info.instructions, "\n")
    tools = await session.list_tools()                               # message 2: tools/list
    for t in tools.tools:
        print(f"TOOL {t.name}")
        print(f"   description: {t.description.splitlines()[0]}")
        print(f"   input schema: {json.dumps(t.inputSchema['properties'])}")
        print(f"   output fields: {list((t.outputSchema or {}).get('properties', {}))}\n")

connected to: okf | protocol 2025-11-25
instructions: Search and answer questions about five AI research papers (GPT-4 report, Sparks of AGI, HELM, Foundation Models, LLM survey). 

TOOL search_okf
   description: Find the passages from the five papers that best match a question (no LLM, about 1-2 s).
   input schema: {"question": {"description": "The question or keywords to search for", "title": "Question", "type": "string"}}
   output fields: ['result']

TOOL ask_okf
   description: Answer a question from the five papers with citations like [papers/helm/...].
   input schema: {"question": {"description": "A question about the five papers", "title": "Question", "type": "string"}}
   output fields: ['answer', 'refused', 'sources', 'model', 'seconds']

TOOL get_concept
   description: Read one topic page or paper section in full, e.g. to follow a citation.
   input schema: {"concept_id": {"description": "A concept id as cited, e.g. papers/helm/4-4-calibration-and-uncertainty", "title": 

Read that output carefully: it's **everything a model will ever know about your tools**: name, description, input schema. That's why the docstrings and `Field(description=...)` matter as much as the code.

## 4. Call a tool: `search_okf` (retrieval only, no LLM)

Message 3: `tools/call`. Each result comes in **two forms**:
- `structuredContent`: JSON matching the output schema → for **code** (a list is wrapped as `{"result": [...]}`)
- `content`: the same thing as text blocks → for **models**

In [4]:
q = "What score did GPT-4 get on the bar exam?"
async with mcp_session() as (session, _):
    t0 = time.perf_counter()
    r = await session.call_tool("search_okf", {"question": q})        # message 3: tools/call
print(f"{time.perf_counter() - t0:.1f} s | isError: {r.isError} | {len(r.content)} text blocks\n")
for p in r.structuredContent["result"]:
    print(f"  [{p['source']['id']}]  {p['source']['label']}  ({len(p['text'])} chars)")
print("\nwhat a model would read (first text block, start):\n" + r.content[0].text[:400])

1.4 s | isError: False | 5 text blocks

  [topics/simulated-exams]  curated Benchmark page: Simulated professional and academic exams  (2190 chars)
  [papers/gpt-4-technical-report/overview]  curated Paper page: GPT-4 Technical Report  (2854 chars)
  [papers/gpt-4-technical-report/4-capabilities-part-1]  GPT-4 Technical Report, 4 Capabilities (part 1 of 3), pp. 4-6  (4592 chars)
  [papers/gpt-4-technical-report/abstract]  GPT-4 Technical Report, Abstract, p. 1  (1015 chars)
  [papers/sparks-of-agi/1-2-organization-of-our-demonstration-part-1]  Sparks of Artificial General Intelligence: Early experiments with GPT-4, 1.2 Organization of our demonstration (part 1 of 4), pp. 8-9  (4655 chars)

what a model would read (first text block, start):
{
  "source": {
    "id": "topics/simulated-exams",
    "kind": "topic",
    "title": "Simulated professional and academic exams",
    "paper": null,
    "pages": null,
    "label": "curated Benchmark page: Simulated professional and academic exams"


## 5. `ask_okf`: retrieval + qwen's answer

Same pipeline as `okf.ask()` in `09`, now behind the server. Notice how small the result is compared with `Answer`: no context, no retrieval details.

In [5]:
async with mcp_session() as (session, _):
    r = await session.call_tool("ask_okf", {"question": q})
a = r.structuredContent
print(a["answer"], "\n")
for s in a["sources"]:
    print(f"  [{s['id']}] {s['label']}")
print(f"\nrefused: {a['refused']} | model: {a['model']} | {a['seconds']} s | result size: {len(r.content[0].text)} chars")

GPT-4 scored 298/400 on the Uniform Bar Exam, which corresponds to the 90th percentile (top 10%) of human test takers. [topics/simulated-exams] [papers/gpt-4-technical-report/4-capabilities-part-1] 

  [topics/simulated-exams] curated Benchmark page: Simulated professional and academic exams
  [papers/gpt-4-technical-report/4-capabilities-part-1] GPT-4 Technical Report, 4 Capabilities (part 1 of 3), pp. 4-6

refused: False | model: ollama/qwen3:4b-instruct-2507-q4_K_M | 9.9 s | result size: 880 chars


## 6. Errors and following a citation: `get_concept`

A wrong id doesn't crash anything: the server answers with `isError: True` and the message. Then a real id from the answer above, to read that source in full.

In [6]:
async with mcp_session() as (session, _):
    bad = await session.call_tool("get_concept", {"concept_id": "papers/nope"})
    print("isError:", bad.isError, "|", bad.content[0].text, "\n")

    cited = a["sources"][0]["id"] if a["sources"] else "topics/simulated-exams"
    good = await session.call_tool("get_concept", {"concept_id": cited})
    print(f"[{cited}] {len(good.structuredContent['text'])} chars:\n")
    print(good.structuredContent["text"][:500], "...")

isError: True | Error executing tool get_concept: No concept 'papers/nope'. Use an id exactly as it appears in a citation. 

[topics/simulated-exams] 2190 chars:

### [topics/simulated-exams] (curated Benchmark page)
# Simulated professional and academic exams

## Summary

The GPT-4 Technical Report evaluates GPT-4 on exams written for humans, with no exam-specific training. The exams were scored with each exam's own rubric, and the score was converted to a percentile of human test takers. The headline result is a simulated Uniform Bar Exam score of 298/400, around the 90th percentile (top 10%), against 213/400 (~10th percentile) for GPT-3.5 (4 Capabiliti ...


## 7. A resource: `okf://status`

Resources are read by address, not called. An app (e.g. the UI) would read this to show whether vectors are on and which model answers.

In [7]:
async with mcp_session() as (session, _):
    resources = await session.list_resources()
    print("resources:", [str(x.uri) for x in resources.resources])
    res = await session.read_resource("okf://status")
print(res.contents[0].text)

resources: ['okf://status']
{
  "concepts": 857,
  "topic_pages": 59,
  "sections": 798,
  "vectors": "on",
  "topic_vectors": true,
  "chunks_mapped": 2900,
  "llm": "ollama/qwen3:4b-instruct-2507-q4_K_M",
  "llm_ok": true,
  "llm_status": "ok"
}


## 8. Preview of stage 3: the same tools inside LangGraph

`langchain-mcp-adapters` turns every MCP tool into a normal LangChain tool. This is the whole bridge between the server and the agent graph we build next.

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({"okf": {"url": URL, "transport": "streamable_http"}})
lc_tools = await client.get_tools()
print([t.name for t in lc_tools])

search_tool = next(t for t in lc_tools if t.name == "search_okf")
out = await search_tool.ainvoke({"question": "How does HELM measure calibration?"})
print(type(out).__name__, "with", len(out), "items | first starts:", str(out[0])[:200])

['search_okf', 'ask_okf', 'get_concept']
list with 4 items | first starts: {'type': 'text', 'text': '{\n  "source": {\n    "id": "papers/helm/overview",\n    "kind": "topic",\n    "title": "Holistic Evaluation of Language Models (HELM)",\n    "paper": null,\n    "pages": nul


## 9. Other ways to use the server

**MCP Inspector** (a web page to click through tools; needs Node.js). Stop the notebook's server first (next cell), then in a terminal:
```
mcp dev okf_mcp/server.py
```

**Claude Desktop** (optional): Settings → Developer → Edit config, add the server over stdio. Use the full path to the `unsloth` env's python (`where python` in an activated terminal shows it):
```json
{
  "mcpServers": {
    "okf": {
      "command": "C:\\path\\to\\envs\\unsloth\\python.exe",
      "args": ["-m", "okf_mcp.server", "--stdio"],
      "cwd": "C:\\Users\\Swapn\\OneDrive\\Documents\\GitHub\\OKF"
    }
  }
}
```
Restart Claude Desktop and it can search your five papers.

In [9]:
# stop the server this notebook started (a server you started yourself in a terminal is left alone)
if server is not None and server.poll() is None:
    server.terminate()
    server.wait(timeout=10)
    print("server stopped")
else:
    print("nothing to stop")

nothing to stop


## What's next

Stage 3: the **OKF agent in LangGraph**. A graph with nodes `check_input → search (via this MCP server) → answer → check_citations`, chat memory in **Postgres**, and **one** qwen call per question (it calls `search_okf`, not `ask_okf`, and writes the answer itself; `ask_okf` stays for outside clients like Claude Desktop).